# SmolVLM-256M: Stop at the End of the Answer

This notebook tests when to stop the decode loop. It compares running all 64 steps with checking for the end token every step, every 4, 8 or 16 steps, and with a fixed short cap.

In [ ]:
import copy
import gc
import json
import re
import subprocess
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, StaticCache
from transformers.modeling_outputs import BaseModelOutput

ROOT = next(p for p in (Path(".."), Path("."), Path("/content/edge-ai-smol-lvm-250-optimization"))
            if (p / "frame.jpg").exists())
IMAGE = ROOT / "frame.jpg"
MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
PROMPT = "Describe any visible defects."
PREPROCESS = {"default": (None, "cpu"), "slow": (False, "cpu"), "fast-cpu": (True, "cpu"), "fast-gpu": (True, "cuda")}


def timed(fn):
    torch.cuda.synchronize()
    t = time.perf_counter()
    out = fn()
    torch.cuda.synchronize()
    return out, (time.perf_counter() - t) * 1e3


def mean_ms(fn, runs=5):
    fn()
    torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(runs):
        fn()
        torch.cuda.synchronize()
    return (time.perf_counter() - t) * 1e3 / runs


def until_eos(ids, eos):
    for i, t in enumerate(ids):
        if t in eos:
            return ids[:i]
    return ids


def set_attn(model, impl):
    if hasattr(model, "set_attn_implementation"):
        model.set_attn_implementation(impl)
    else:
        for c in (model.config.text_config, model.config.vision_config):
            c._attn_implementation = impl


def load(attn=None, fast=None):
    kw = {} if attn is None else {"attn_implementation": attn}
    proc = AutoProcessor.from_pretrained(MODEL_ID, **({} if fast is None else {"use_fast": fast}))
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.float16, **kw).to("cuda").eval()
    return model, proc


def eos_ids(tok):
    return {tok.eos_token_id, tok.convert_tokens_to_ids("<end_of_utterance>")}


def chat_text(proc, prompt):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    return proc.apply_chat_template(msgs, add_generation_prompt=True)


def make_inputs(proc, image, prompt, image_device="cpu", size=None, extra=None):
    img = Image.open(image).convert("RGB") if isinstance(image, (str, Path)) else image
    dev = {} if image_device == "cpu" else {"device": image_device}
    ik = {**({} if size is None else {"size": {"longest_edge": size}}), **dev, **(extra or {})}
    text = chat_text(proc, prompt)
    try:
        out = proc(text=text, images=[img], return_tensors="pt", images_kwargs=ik)
    except (TypeError, ValueError):
        out = proc(text=text, images=[img], return_tensors="pt", **ik)
    out = out.to("cuda")
    out["pixel_values"] = out["pixel_values"].to(torch.float16)
    return out


def bare_generate(model, inputs, k, stop=True):
    extra = {} if stop else {"min_new_tokens": k}
    with torch.inference_mode():
        return model.generate(**inputs, max_new_tokens=k, do_sample=False, **extra)


class GraphDecoder:
    def __init__(self, model, max_new=64, max_len=4096, own_mask=False):
        self.model, self.max_new, self.max_len = model, max_new, max_len
        self.cache = StaticCache(config=model.config.text_config, max_cache_len=max_len)
        dev = model.device
        self.ids = torch.zeros(1, 1, dtype=torch.long, device=dev)
        self.pos = torch.zeros(1, dtype=torch.long, device=dev)
        self.idx = torch.zeros(1, dtype=torch.long, device=dev)
        self.out = torch.zeros(max_new, dtype=torch.long, device=dev)
        self.graph = None
        self.steps = max_new
        self.own_mask = own_mask or model.config.text_config._attn_implementation == "eager"

    def mask(self):
        allowed = torch.arange(self.max_len, device=self.pos.device) <= self.pos[:, None]
        dt = self.model.dtype
        return ((~allowed).to(dt) * torch.finfo(dt).min)[:, None, None, :]

    def step(self):
        extra = {"attention_mask": self.mask()} if self.own_mask else {}
        h = self.model.model.text_model(
            input_ids=self.ids, cache_position=self.pos, past_key_values=self.cache, use_cache=True, **extra
        ).last_hidden_state
        nxt = self.model.lm_head(h[:, -1]).argmax(-1, keepdim=True)
        self.ids.copy_(nxt)
        self.out.index_copy_(0, self.idx, nxt.view(1))
        self.pos.add_(1)
        self.idx.add_(1)

    @torch.inference_mode()
    def capture(self):
        s = torch.cuda.Stream()
        s.wait_stream(torch.cuda.current_stream())
        with torch.cuda.stream(s):
            for _ in range(3):
                self.pos.fill_(self.max_len - 1)
                self.idx.zero_()
                self.step()
        torch.cuda.current_stream().wait_stream(s)
        self.pos.fill_(self.max_len - 1)
        self.idx.zero_()
        self.graph = torch.cuda.CUDAGraph()
        with torch.cuda.graph(self.graph):
            self.step()

    @torch.inference_mode()
    def prefill(self, inputs):
        self.cache.reset()
        n = inputs["input_ids"].shape[1]
        logits = self.model(
            **inputs, past_key_values=self.cache, use_cache=True,
            cache_position=torch.arange(n, device=self.model.device),
        ).logits
        first = logits[:, -1].argmax(-1, keepdim=True)
        self.ids.copy_(first)
        self.pos.fill_(n)
        self.idx.zero_()
        return first

    @torch.inference_mode()
    def decode(self, use_graph=True, eos=None):
        self.steps = self.max_new
        for i in range(self.max_new):
            if use_graph:
                self.graph.replay()
            else:
                self.step()
            if eos and self.ids.item() in eos:
                self.steps = i + 1
                break
        return self.out.clone()

    def generate(self, inputs, eos, use_graph=True):
        first = self.prefill(inputs)
        ids = torch.cat([first.view(1), self.decode(use_graph, eos)]).tolist()
        return until_eos(ids, eos)


def time_decode(dec, inputs, eos, runs=5, use_graph=True, stop=False):
    pre, dec_ms, ids = [], [], None
    for i in range(runs + 1):
        first, p = timed(lambda: dec.prefill(inputs))
        out, d = timed(lambda: dec.decode(use_graph, eos if stop else None))
        if i:
            pre.append(p)
            dec_ms.append(d)
        ids = until_eos(torch.cat([first.view(1), out]).tolist(), eos)
    return {"prefill_ms": float(np.mean(pre)), "decode_ms": float(np.mean(dec_ms)), "steps": dec.steps, "ids": ids}


def fit_len(n, max_new):
    return -(-(n + max_new + 1) // 64) * 64

In [ ]:
def decode_every(dec, eos, every):
    for i in range(dec.max_new):
        dec.graph.replay()
        if every and (i + 1) % every == 0 and any(t in eos for t in dec.out[: i + 1].tolist()):
            return i + 1
    return dec.max_new


def strategy(model, inputs, eos, name, n, every, runs=5):
    dec = GraphDecoder(model, n, 1024)
    dec.capture()
    pre, dec_ms, steps = [], [], None
    for i in range(runs + 1):
        first, p = timed(lambda: dec.prefill(inputs))
        steps, d = timed(lambda: decode_every(dec, eos, every))
        if i:
            pre.append(p)
            dec_ms.append(d)
    ids = until_eos([first.item()] + dec.out[:steps].tolist(), eos)
    return {"strategy": name, "steps": steps, "decode ms": round(float(np.mean(dec_ms)), 1),
            "total ms": round(float(np.mean(pre) + np.mean(dec_ms)), 1), "ids": ids}


model, proc = load()
inputs = make_inputs(proc, IMAGE, PROMPT)
eos = eos_ids(proc.tokenizer)
results = [strategy(model, inputs, eos, "fixed", 64, None), strategy(model, inputs, eos, "stop1", 64, 1),
           strategy(model, inputs, eos, "stop4", 64, 4), strategy(model, inputs, eos, "stop8", 64, 8),
           strategy(model, inputs, eos, "stop16", 64, 16), strategy(model, inputs, eos, "cap16", 16, None),
           strategy(model, inputs, eos, "cap32", 32, None)]
base = results[0]["total ms"]
table = pd.DataFrame(results)
table["saved ms"] = (base - table["total ms"]).round(1)
table["complete answer"] = [r["ids"] == results[0]["ids"] for r in results]
table.drop(columns="ids")

## Log of previous runs

The cells below are the earlier notebook with their outputs. They call the scripts that were replaced by the code above, so they are not run again.

In [21]:
REPO = "./edge-ai-smol-lvm-250-optimization"
!python {REPO}/eos_stop.py --image {REPO}/frame.jpg --max-len 1024

Loading weights: 100% 471/471 [00:00<00:00, 969.33it/s] 
strategy  steps  decode ms  total ms  saved ms  complete answer
answer tokens: 9
fixed        64      519.0     779.8       0.0  True
stop1         9       79.2     339.6     440.2  True
stop4        12      103.4     365.8     414.0  True
stop8        16      136.4     394.6     385.2  True
stop16       16      136.6     401.3     378.6  True
cap16        16      136.1     404.1     375.8  True
cap32        32      264.8     532.5     247.3  True


## Result

The test ran on a Colab GPU with one image. The answer has 9 tokens. All strategies give the complete answer.

| Strategy | Steps | Decode (ms) | Total (ms) | Saved (ms) | Complete answer |
|---|---|---|---|---|---|
| fixed | 64 | 519.0 | 779.8 | 0.0 | True |
| stop1 | 9 | 79.2 | 339.6 | 440.2 | True |
| stop4 | 12 | 103.4 | 365.8 | 414.0 | True |
| stop8 | 16 | 136.4 | 394.6 | 385.2 | True |
| stop16 | 16 | 136.6 | 401.3 | 378.6 | True |
| cap16 | 16 | 136.1 | 404.1 | 375.8 | True |
| cap32 | 32 | 264.8 | 532.5 | 247.3 | True |

Total is the prefill plus the decode.

- `stop1` is the best strategy. It cuts the decode from 519 ms to 79 ms. It cuts the total from 780 ms to 340 ms (56% less).
- The check costs almost nothing. A line through the runs gives about 8.0 ms for each step plus about 7 ms of fixed time. The line predicts 79.8 ms for `stop1`. The measured value is 79.2 ms.
- Each wasted step costs 8 ms, and a check costs less than 1 ms. Use K = 1.
- `cap16` and `cap32` give no gain over `stop1`. They can also cut a longer answer.
- Treat differences under about 10 ms in the total column as noise. The prefill part varies by about 5 ms.
- With `fast-gpu`, the request is about 378 ms with `stop1`. It is 823 ms with 64 steps (estimate from separate runs). Prefill is now about 75% of the request.
- The hybrid attention gain is now small. It is about 4 ms for a 9-token answer.
- The test used one image and one prompt. The saving changes with the answer length.
- `GraphDecoder.decode` and `generate` now use `stop1`. `bench.py` prints the time with and without the stop.